# 03. confidence 기반 캐스케이드

## 배경

Jev 의 답에는 probability 가 붙어 있습니다. 이 숫자를 버리고 답만 쓰면 "틀릴 것 같은 답"과 "확실한 답"을 똑같이 취급하게 됩니다. 캐스케이드는 확신이 높으면 Jev 의 답을 그대로 쓰고, 낮으면 더 느리지만 더 신중한 쪽으로 넘기는 구조입니다.

## 직관

병원 접수 창구를 생각하면 됩니다. 접수 직원은 대부분의 환자를 바로 맞는 과로 보냅니다. 증상이 애매하면 의사에게 묻습니다. 모든 환자를 의사가 직접 분류하면 정확하지만 느리고 비쌉니다.

```
            확신 높음 ─────────────▶ Jev 답 사용 (빠르고 저렴)
Jev 판단 ──┤
            확신 낮음 ─▶ LLM 에 위임 ─▶ 되돌릴 수 없는 작업이면 사람에게 확인
```

TypeSafe 는 Jev 의 probability 가 calibration 이 되어 있다고 설명합니다. 80% 라고 답한 판단을 여러 개 모으면 그중 약 80% 가 맞는다는 뜻입니다. 판단 하나만 놓고 보면 맞을 수도 틀릴 수도 있습니다.

In [1]:
import sys
sys.path.insert(0, "..")  # 프로젝트 루트의 jev_agent 패키지를 불러오기 위함

from dotenv import load_dotenv
load_dotenv("../.env")

from jev_agent.jev import JevClient, choice, noul, score

jev = JevClient()

## 명확한 질의와 애매한 질의

같은 질문을 두 종류의 질의에 던져 `confidence` 가 어떻게 달라지는지 봅니다.

In [2]:
import pandas as pd
from jev_agent.middleware import NO_TOOL
from jev_agent.tools import tool_catalog

OPTIONS = {**tool_catalog(), NO_TOOL: "도구가 필요 없다. 인사나 잡담이다."}
QUESTION = {"tool": choice("사용자 요청을 처리하기 위해 실행해야 할 도구는 무엇인가?", OPTIONS)}

queries = [
    "A1001 지금 어디쯤 왔어요?",           # 명확
    "C001 적립금 얼마 남았어요?",           # 명확
    "A1002 그거 어떻게 됐어요?",            # 조회인지 배송인지 애매
    "A1003 이거 그냥 안 할래요.",           # 취소인지 환불인지 애매
    "이어폰 문제 있는데 어떻게 하죠?",       # 환불? 안내? 상담원?
]
rows = []
for query in queries:
    answer = jev.decide({"user_request": query}, QUESTION)["tool"]
    rows.append({"질의": query, "선택": answer["choice"], "confidence": round(answer["confidence"], 2)})
pd.DataFrame(rows)

,질의,선택,confidence
0,A1001 지금 어디쯤 왔어요?,track_shipping,0.99
1,C001 적립금 얼마 남았어요?,check_points,1.00
2,A1002 그거 어떻게 됐어요?,search_order,0.64
3,A1003 이거 그냥 안 할래요.,cancel_order,0.82
4,이어폰 문제 있는데 어떻게 하죠?,search_faq,0.59


## 캐스케이드 1: Jev → LLM

confidence 가 기준보다 낮으면 LLM 에게 구조화 출력으로 다시 묻습니다. 기준값 `MIN_CONFIDENCE` 는 정해진 정답이 없습니다. 05번에서 기준을 바꿔 가며 정확도와 비용이 어떻게 움직이는지 측정합니다.

In [3]:
from typing import Literal
from pydantic import BaseModel
from jev_agent.agent import build_chat_model

MIN_CONFIDENCE = 0.7  # 설명을 위해 높게 잡았다. 애매한 질의 두 건이 LLM 으로 넘어간다.
ToolName = Literal[tuple(OPTIONS)]  # option 밖의 값은 스키마에서 거부된다

class ToolPick(BaseModel):
    tool: ToolName

llm_picker = build_chat_model().with_structured_output(ToolPick)
tool_descriptions = "\n".join(f"- {name}: {description}" for name, description in OPTIONS.items())

def pick_with_cascade(user_request: str) -> dict:
    answer = jev.decide({"user_request": user_request}, QUESTION)["tool"]
    if answer["confidence"] >= MIN_CONFIDENCE:
        return {"tool": answer["choice"], "decided_by": "Jev", "confidence": answer["confidence"]}
    picked = llm_picker.invoke(
        f"다음 도구 중 사용자 요청에 필요한 것 하나를 고르세요.\n{tool_descriptions}\n\n요청: {user_request}"
    )
    return {"tool": picked.tool, "decided_by": "LLM", "confidence": answer["confidence"]}

pd.DataFrame([{"질의": query, **pick_with_cascade(query)} for query in queries])

,질의,tool,decided_by,confidence
0,A1001 지금 어디쯤 왔어요?,track_shipping,Jev,1.00
1,C001 적립금 얼마 남았어요?,check_points,Jev,1.00
2,A1002 그거 어떻게 됐어요?,search_order,LLM,0.61
3,A1003 이거 그냥 안 할래요.,cancel_order,Jev,0.81
4,이어폰 문제 있는데 어떻게 하죠?,search_faq,LLM,0.57


## 캐스케이드 2: 되돌릴 수 없는 작업은 사람에게

환불과 주문 취소는 되돌릴 수 없습니다. 이런 도구는 실행 직전에 Jev 에게 "사용자가 이 작업을 직접 요청했는가"를 `noul` 로 묻고, probability 에 따라 셋으로 나눕니다.

| probability | 처리 |
|---|---|
| 0.93 이상 | 바로 실행 |
| 0.15 이하 | 실행하지 않고 거절 메시지 반환 |
| 그 사이 | `interrupt()` 로 멈추고 사람의 승인을 기다림 |

0.93 이라는 기준은 이 데모의 문장들에 맞춰 고른 값입니다. 분명한 환불 요청은 0.97 안팎, 망설이는 요청은 0.85~0.90 으로 나와서 둘 사이 간격이 0.1 이 채 되지 않습니다. 실제 서비스에서는 자기 데이터로 probability 분포를 확인한 뒤 기준을 정해야 합니다.

먼저 게이트가 쓰는 질문만 따로 떼어 probability 를 봅니다.

In [4]:
from jev_agent.middleware import JevRiskGateMiddleware

gate_question = JevRiskGateMiddleware._question()
proposed = {"proposed_action": "request_refund", "arguments": {"order_id": "A1002", "reason": "불량"}}

for user_request in [
    "A1002 불량이라 환불해 주세요.",      # 직접 요청
    "A1002 환불해야 하나 싶은데… 잘 모르겠네요. 일단 환불 접수해 주세요",  # 요청은 했지만 망설임
    "A1002 이어폰이 좀 별로네요.",         # 불만은 있지만 환불 요청은 아님
    "A1002 배송 완료됐나요?",              # 환불과 무관
]:
    probability = jev.decide({"user_request": user_request, **proposed}, gate_question)["requested"]["noul"]
    print(f"{probability:.2f}  {user_request}")

0.97  A1002 불량이라 환불해 주세요.


0.84  A1002 환불해야 하나 싶은데… 잘 모르겠네요. 일단 환불 접수해 주세요


0.15  A1002 이어폰이 좀 별로네요.


0.03  A1002 배송 완료됐나요?


## 에이전트에 게이트 붙이기

`interrupt()` 를 쓰려면 체크포인터와 `thread_id` 가 필요합니다. 멈춘 그래프는 같은 `thread_id` 로 `Command(resume=...)` 를 보내 이어갑니다.

`interrupt()` 앞의 코드는 재개할 때 한 번 더 실행됩니다. 게이트에서 `interrupt()` 앞에 있는 것은 Jev 호출뿐이고 이 호출은 아무것도 바꾸지 않으므로 다시 실행돼도 안전합니다. 환불 실행은 `interrupt()` 뒤에 있습니다.

In [5]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from jev_agent import tools as shop
from jev_agent.agent import SUPPORT_SYSTEM_PROMPT

agent = create_deep_agent(
    model=build_chat_model(),
    tools=shop.TOOLS,
    system_prompt=SUPPORT_SYSTEM_PROMPT,
    middleware=[JevRiskGateMiddleware(jev, risky_tools=shop.RISKY_TOOLS)],
    checkpointer=InMemorySaver(),
)

def run(text: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    state = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
    for decision in state.get("jev_decisions", []):
        print("Jev:", decision["title"], "→", decision["verdict"], decision.get("probabilities"))
    if "__interrupt__" in state:
        print("멈춤. 승인 요청:", state["__interrupt__"][0].value["tool"], state["__interrupt__"][0].value["args"])
    else:
        print("답변:", state["messages"][-1].content)
    return state, config

state, config = run("A1002 불량이라 환불해 주세요.", "clear-request")
print("주문 상태:", shop.ORDERS["A1002"]["status"])

Jev: 위험 도구 게이트: request_refund → 자동 승인 {'requested': 0.97, 'not_requested': 0.030000000000000027}
답변: 환불 접수 완료되었습니다.  
주문 A1002, 사유는 상품 불량이며 환불 금액은 129,000원입니다.
주문 상태: 환불접수


이번에는 망설이는 요청입니다. LLM 은 "접수해 주세요"를 보고 환불 도구를 부르지만, Jev 는 앞의 망설임 때문에 probability 를 중간 구간으로 내놓고 그래프가 멈춥니다. LLM 응답은 실행마다 조금씩 달라서, 환불 도구를 부르지 않고 되묻는 경우에는 게이트가 동작하지 않고 일반 답변이 나옵니다.

In [6]:
shop.ORDERS["A1002"]["status"] = "배송완료"  # 앞 셀에서 바뀐 상태를 되돌림

state, config = run("A1002 환불해야 하나 싶은데… 잘 모르겠네요. 일단 환불 접수해 주세요", "ambiguous-request")

if "__interrupt__" in state:
    resumed = agent.invoke(Command(resume={"approved": False}), config)  # 사람이 거절
    print("거절 후 답변:", resumed["messages"][-1].content)
print("주문 상태:", shop.ORDERS["A1002"]["status"])

멈춤. 승인 요청: request_refund {'order_id': 'A1002', 'reason': '고객 요청에 따른 환불 접수'}


거절 후 답변: 환불 접수는 바로 진행되지 않았습니다.  
현재 환불 요청 처리는 승인 필요로 막혀 있어요.

원하시면 제가 대신 상담원 연결을 요청해 드릴게요.
주문 상태: 배송완료


## 정리

- `confidence` 와 `noul` probability 는 분기 조건으로 쓸 수 있습니다.
- 확신이 높으면 Jev, 낮으면 LLM, 되돌릴 수 없는 작업이 애매하면 사람으로 넘깁니다.
- 기준값은 서비스마다 다릅니다. 잘못 실행했을 때의 비용이 클수록 승인 기준을 높입니다.
- 권한이나 결제 같은 결정을 모델 판단에만 맡기면 안 됩니다. 이 게이트는 서버 쪽 검증을 대신하는 것이 아니라 그 앞에 한 겹을 더하는 것입니다.